In [ ]:
# Colab setup: install pyedautils; elsewhere load plotly.js from a CDN for the plots
import sys
if "google.colab" in sys.modules:
    %pip install -q "pyedautils>=0.0.33"
else:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"

# Electricity Household

```{admonition} Goal
:class: tip
Visualize household electricity consumption patterns over time using a line chart. This provides an overview of how electricity usage varies across the measurement period.
```

```{admonition} Data Basis
:class: note
The dataset `eboBookEleMeter.csv` contains cumulative electricity meter readings at 15-minute intervals (columns: `timestamp`, `eleMeterVal`; separator `;`). The meter counts **Wh** (it rises by about 1,000,000 in 2015, i.e. about 1,000 kWh per year). We compute consumption by differencing the cumulative values and convert it to kWh.
```

## Data Preparation

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/eboBookEleMeter.csv"
raw = pd.read_csv(url, sep=";")

# Parse timestamp and set as index
raw["timestamp"] = pd.to_datetime(raw["timestamp"])
raw = raw.set_index("timestamp")

# Consumption per 15 minutes from the cumulative meter readings, Wh -> kWh
consumption = raw.diff() / 1000

# Hourly sums for a cleaner chart (complete hours only: all 4 quarter-hours present)
hourly = consumption.resample("h").sum(min_count=4).dropna()

hourly.head()

## Visualization

In [ ]:
from pyedautils.plots import plot_timeseries

fig = plot_timeseries(
    hourly,
    columns=["eleMeterVal"],
    title="Household Electricity Consumption",
    ylab="Electricity [kWh/h]",
)
fig.show()

::::{dropdown} Customization
You can change the resampling frequency to see different time granularities:

```python
# Daily totals [kWh/d], complete days only
daily = consumption.resample("D").sum(min_count=96).dropna()
```
::::